In [4]:
# Defining project paths
BASE_DIR = "D:/Capstone/capstone_repo"
DATA_DIR = f"{BASE_DIR}/data"
NB_DIR = f"{BASE_DIR}/notebooks"

import os
os.makedirs(DATA_DIR, exist_ok=True)

In [2]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
from shapely.geometry import Point
from shapely.ops import nearest_points

# Analysis

In [9]:
healthcare = pd.read_csv(f"{DATA_DIR}/processed/casablanca_healthcare_cleaned.csv")
stops = pd.read_csv(f"{DATA_DIR}/processed/casablanca_transport_stops_cleaned.csv")

In [10]:
# Convert to GeoDataFrames
healthcare_gdf = gpd.GeoDataFrame(
    healthcare,
    geometry=gpd.points_from_xy(healthcare.lon, healthcare.lat),
    crs="EPSG:4326"
)

stops_gdf = gpd.GeoDataFrame(
    stops,
    geometry=gpd.points_from_xy(stops.lon, stops.lat),
    crs="EPSG:4326"
)


In [11]:
# Project to UTM
healthcare_gdf = healthcare_gdf.to_crs(epsg=32629)
stops_gdf = stops_gdf.to_crs(epsg=32629)

In [12]:
# Compute nearest stop distance
def nearest_distance(point, stops_geom):
    nearest_geom = stops_geom.geometry.unary_union
    nearest_point = nearest_points(point, nearest_geom)[1]
    return point.distance(nearest_point)

healthcare_gdf['distance_to_stop_m'] = healthcare_gdf.geometry.apply(
    lambda x: nearest_distance(x, stops_gdf)
)

C:\Users\afafb\AppData\Local\Temp\ipykernel_14444\1660183459.py:3: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  nearest_geom = stops_geom.geometry.unary_union


In [13]:
# Convert meters to minutes
healthcare_gdf['walking_time_min'] = healthcare_gdf['distance_to_stop_m'] / 83.3

In [14]:
# Results
healthcare_gdf[['name', 'distance_to_stop_m', 'walking_time_min']].head()


,name,distance_to_stop_m,walking_time_min
0,Clinique Badr مصحة بدر,312.197016,3.747863
1,clinique dentaire casablanca (cdc),30.472442,0.365816
2,Centre consultation et traitement dentaires,208.219746,2.499637
3,Clinique d'accouchement,149.184204,1.790927
4,Hôpital Sidi Othmane,202.572789,2.431846


In [15]:
healthcare_gdf['accessible_5min'] = healthcare_gdf['walking_time_min'] <= 5
healthcare_gdf['accessible_5min'].value_counts(normalize=True) * 100

accessible_5min
True     99.047619
False     0.952381
Name: proportion, dtype: float64

# Visualization

In [16]:
# Convert your data back to WGS84 (for mapping)
healthcare_map = healthcare_gdf.to_crs(epsg=4326)
stops_map = stops_gdf.to_crs(epsg=4326)

In [17]:
# create an interactive map using folium
import folium

# Center map on Casablanca
m = folium.Map(location=[33.5731, -7.5898], zoom_start=12)

# Add transport stops (blue)
for _, row in stops_map.iterrows():
    folium.CircleMarker(
        location=[row['lat'], row['lon']],
        radius=2,
        color='blue',
        fill=True,
        fill_opacity=0.5,
        popup=row['name']
    ).add_to(m)

# Add healthcare facilities with accessibility colors
for _, row in healthcare_map.iterrows():
    if row['walking_time_min'] <= 5:
        color = 'green'   # good access
    elif row['walking_time_min'] <= 10:
        color = 'orange'  # medium access
    else:
        color = 'red'     # poor access

    folium.Marker(
        location=[row['lat'], row['lon']],
        popup=f"{row['name']}<br>Time: {row['walking_time_min']:.2f} min",
        icon=folium.Icon(color=color)
    ).add_to(m)

m


In [18]:
m.save("casablanca_accessibility_map.html")

# Analysis by district

In [9]:
districts = gpd.read_file(f"{DATA_DIR}/raw/gadm/gadm41_MAR_4.shp") 
districts.head()

,GID_4,GID_0,COUNTRY,GID_1,NAME_1,GID_2,NAME_2,GID_3,NAME_3,NAME_4,VARNAME_4,TYPE_4,ENGTYPE_4,CC_4,geometry
0,MAR.1.1.1.1_1,MAR,Morocco,MAR.1_1,Chaouia - Ouardigha,MAR.1.1_1,Ben Slimane,MAR.1.1.1_1,Ben Slimane,Ahlaf,NA,Commune Rural,Rural Commune,NA,"POLYGON ((-7.21601 33.33524, -7.20212 33.34536..."
1,MAR.1.1.1.2_1,MAR,Morocco,MAR.1_1,Chaouia - Ouardigha,MAR.1.1_1,Ben Slimane,MAR.1.1.1_1,Ben Slimane,Ain Tizgha,NA,Commune Rural,Rural Commune,NA,"POLYGON ((-7.01937 33.57966, -7.02208 33.57954..."
2,MAR.1.1.1.3_1,MAR,Morocco,MAR.1_1,Chaouia - Ouardigha,MAR.1.1_1,Ben Slimane,MAR.1.1.1_1,Ben Slimane,Fdalate,NA,Commune Rural,Rural Commune,NA,"POLYGON ((-7.25532 33.68698, -7.24397 33.66458..."
3,MAR.1.1.1.4_1,MAR,Morocco,MAR.1_1,Chaouia - Ouardigha,MAR.1.1_1,Ben Slimane,MAR.1.1.1_1,Ben Slimane,Mellila,NA,Commune Rural,Rural Commune,NA,"POLYGON ((-6.96569 33.36103, -6.96891 33.35316..."
4,MAR.1.1.1.5_1,MAR,Morocco,MAR.1_1,Chaouia - Ouardigha,MAR.1.1_1,Ben Slimane,MAR.1.1.1_1,Ben Slimane,Moualine El Ghaba,NA,Commune Rural,Rural Commune,NA,"POLYGON ((-7.12737 33.72615, -7.1226 33.71605,..."


In [10]:
casablanca_districts = districts[districts['NAME_2'].str.contains("Casablanca", case=False, na=False)]

In [11]:
casablanca_districts = casablanca_districts.to_crs(epsg=4326)


In [8]:
casablanca_districts.head()

,GID_3,GID_0,COUNTRY,GID_1,NAME_1,NL_NAME_1,GID_2,NAME_2,NL_NAME_2,NAME_3,VARNAME_3,NL_NAME_3,TYPE_3,ENGTYPE_3,CC_3,HASC_3,geometry
91,MAR.5.1.1_1,MAR,Morocco,MAR.5_1,Grand Casablanca,NA,MAR.5.1_1,Casablanca,NA,Bouskoura,NA,NA,Cercle,District,NA,NA,"POLYGON ((-7.68806 33.41307, -7.69137 33.41693..."
92,MAR.5.1.2_1,MAR,Morocco,MAR.5_1,Grand Casablanca,NA,MAR.5.1_1,Casablanca,NA,Mediouna,NA,NA,Cercle,District,NA,NA,"POLYGON ((-7.52772 33.40506, -7.53161 33.40764..."
93,MAR.5.1.3_1,MAR,Morocco,MAR.5_1,Grand Casablanca,NA,MAR.5.1_1,Casablanca,NA,NA (Ahl Laghlam),NA,NA,Unknown,Unknown,NA,NA,"POLYGON ((-7.49482 33.55348, -7.50156 33.55503..."
94,MAR.5.1.4_1,MAR,Morocco,MAR.5_1,Grand Casablanca,NA,MAR.5.1_1,Casablanca,NA,NA (Ain Chock),NA,NA,Unknown,Unknown,NA,NA,"POLYGON ((-7.5775 33.52416, -7.58148 33.5237, ..."
95,MAR.5.1.5_1,MAR,Morocco,MAR.5_1,Grand Casablanca,NA,MAR.5.1_1,Casablanca,NA,NA (Ain Harrouda),NA,NA,Unknown,Unknown,NA,NA,"POLYGON ((-7.51042 33.6343, -7.51042 33.63458,..."


In [ ]:
# Assign each hospital to a district
healthcare_with_district = gpd.sjoin(
    healthcare_map,
    casablanca_districts,
    how="left",
    predicate="within"
)


In [23]:
district_accessibility = healthcare_with_district.groupby('NAME_3')['walking_time_min'].mean().reset_index()

district_accessibility.sort_values(by='walking_time_min')


KeyError: 'NAME_3'